# A1 – Calidad de los datos

En este notebook revisamos los datos antes de analizarlos. Primero los miramos tal como vinieron (`cargar_crudo`). Al final comprobamos que la versión limpia (`cargar_limpio`) aplica lo que decidimos en `DECISIONES.md`.

1. Cómo es cada tabla
2. Cruces entre tablas
3. Los tres problemas más importantes
4. Métricas de calidad de cada tabla
5. ¿Los datos dicen lo mismo que el enunciado?
6. Cómo se relacionan las tablas
7. Comprobación de los datos limpios

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
from src.carga import cargar_crudo, cargar_limpio, RAW, FECHAS

d = cargar_crudo()
cli = d["clientes"]
trx = d["transacciones"]
prd = d["productos"]
dev = d["devoluciones"]
crm = d["interacciones_crm"]
cam = d["campanas_marketing"]

for nombre, tabla in d.items():
    print(f"{nombre:20s} {tabla.shape[0]:6d} filas  {tabla.shape[1]:2d} columnas")

clientes               8000 filas   7 columnas
transacciones         50000 filas  10 columnas
productos               318 filas   7 columnas
devoluciones           6724 filas   6 columnas
interacciones_crm      4684 filas   7 columnas
campanas_marketing     8820 filas   8 columnas


## 1. Cómo es cada tabla

Leemos cada archivo como texto, sin convertir nada, para ver los valores exactamente como vienen. En cada columna buscamos:

- celdas vacías,
- datos faltantes escritos como texto ("null", "N/A", "No especificado"…),
- filas repetidas,
- valores mínimos y máximos,
- fechas que no se puedan leer.

In [2]:
NULOS_DISFRAZADOS = ["nan", "null", "none", "n/a", "na", "-", "?", "no especificado", "desconocido"]

for nombre in d:
    t = pd.read_csv(RAW / f"{nombre}.csv", dtype=str, keep_default_na=False)
    id_col = t.columns[0]
    print(f"\n=== {nombre}: {len(t)} filas")
    print(f"duplicados exactos: {t.duplicated().sum()}  |  {id_col} repetidos: {t[id_col].duplicated().sum()}")

    for col in t.columns:
        s = t[col].str.strip()
        vacios = (s == "").sum()
        disfrazados = s.str.lower().isin(NULOS_DISFRAZADOS).sum()
        linea = f"  {col:22s} vacíos={vacios:4d}  disfrazados={disfrazados:4d}  únicos={s.nunique():6d}"

        num = pd.to_numeric(s, errors="coerce")
        if num.notna().mean() > 0.9:
            linea += f"  | min={num.min():g}  mediana={num.median():g}  max={num.max():g}"
        elif col in FECHAS.get(nombre, []):
            f = pd.to_datetime(s, errors="coerce", format="%Y-%m-%d")
            linea += f"  | {f.min().date()} a {f.max().date()}  (no se pueden leer: {f.isna().sum() - vacios})"
        elif s.nunique() <= 10:
            linea += f"  | {s.value_counts().to_dict()}"
        print(linea)


=== clientes: 8000 filas
duplicados exactos: 0  |  cliente_id repetidos: 0
  cliente_id             vacíos=   0  disfrazados=   0  únicos=  8000
  edad                   vacíos=   0  disfrazados=   0  únicos=    55  | min=18  mediana=45  max=72
  genero                 vacíos=   0  disfrazados= 433  únicos=     3  | {'F': 3834, 'M': 3733, 'No especificado': 433}
  ciudad                 vacíos=   0  disfrazados=   0  únicos=     8  | {'Caracas': 3170, 'Maracaibo': 1457, 'Valencia': 1201, 'Barquisimeto': 793, 'Mérida': 425, 'Puerto La Cruz': 414, 'Maturín': 314, 'San Cristóbal': 226}
  segmento               vacíos=   0  disfrazados=   0  únicos=     5  | {'Esporádico': 2827, 'Frecuente': 2422, 'Inactivo': 1136, 'Nuevo': 810, 'Premium': 805}
  fecha_registro         vacíos=   0  disfrazados=   0  únicos=   730  | 2023-01-01 a 2024-12-31  (no se pueden leer: 0)
  canal_adquisicion      vacíos=   0  disfrazados=   0  únicos=     6  | {'Redes_sociales': 2036, 'Tienda_física': 2032, 'Orgán

  fecha                  vacíos=   0  disfrazados=   0  únicos=   731  | 2023-01-01 a 2024-12-31  (no se pueden leer: 0)
  producto_id            vacíos=   0  disfrazados=   0  únicos=   318
  cantidad               vacíos=   0  disfrazados=   0  únicos=     5  | min=1  mediana=1  max=5
  precio_unitario        vacíos=   0  disfrazados=   0  únicos=   317  | min=4.04  mediana=116.6  max=1186.64
  descuento              vacíos=   0  disfrazados=   0  únicos=   311  | min=0  mediana=0.029  max=0.345
  monto_neto             vacíos=   0  disfrazados=   0  únicos= 24840  | min=3.1  mediana=163.99  max=5840.92
  canal                  vacíos=   0  disfrazados=   0  únicos=     3  | {'tienda_fisica': 22343, 'online': 19220, 'app_movil': 8437}
  metodo_pago            vacíos=   0  disfrazados=   0  únicos=     5  | {'tarjeta_credito': 14984, 'tarjeta_debito': 14026, 'efectivo': 10916, 'transferencia': 6024, 'billetera_digital': 4050}

=== productos: 318 filas
duplicados exactos: 0  |  product

duplicados exactos: 0  |  campana_id repetidos: 0
  campana_id             vacíos=   0  disfrazados=   0  únicos=  8820
  cliente_id             vacíos=   0  disfrazados=   0  únicos=  4800
  tipo_campana           vacíos=   0  disfrazados=   0  únicos=     5  | {'email_descuento': 3099, 'push_reactivacion': 1768, 'email_fidelizacion': 1748, 'sms_oferta_flash': 1345, 'social_ads': 860}
  fecha_envio            vacíos=   0  disfrazados=   0  únicos=   731  | 2023-01-01 a 2024-12-31  (no se pueden leer: 0)
  canal_campana          vacíos=   0  disfrazados=   0  únicos=     4  | {'email': 4847, 'push_app': 1768, 'sms': 1345, 'redes_sociales': 860}
  abierto                vacíos=   0  disfrazados=   0  únicos=     2  | min=0  mediana=0  max=1
  convertido             vacíos=   0  disfrazados=   0  únicos=     2  | min=0  mediana=0  max=1
  descuento_ofrecido     vacíos=   0  disfrazados=   0  únicos=     6  | min=0  mediana=0.1  max=0.25


**Resultado:**
- No hay celdas vacías ni filas repetidas en ninguna tabla.
- El único dato faltante escrito como texto es el género "No especificado", en 433 clientes.
- Todas las fechas van del 1 de enero de 2023 al 31 de diciembre de 2024 (24 meses).
- Hay 318 productos, no 320 como dice el enunciado.

### Valores fuera de lo normal en cada campo numérico

Para cada campo numérico contamos cuántos valores quedan fuera del rango normal según la regla del rango intercuartílico (IQR): por debajo de Q1 − 1,5·IQR o por encima de Q3 + 1,5·IQR. Junto al conteo mostramos el mínimo y el máximo, para ver si algún valor es imposible y no solo raro.

In [3]:
def atipicos_iqr(s):
    q1, q3 = s.quantile([0.25, 0.75])
    rango = q3 - q1
    return ((s < q1 - 1.5 * rango) | (s > q3 + 1.5 * rango)).sum()

campos = {
    "clientes": ["edad"],
    "transacciones": ["cantidad", "precio_unitario", "descuento", "monto_neto"],
    "productos": ["costo", "precio_lista", "margen_bruto_pct"],
    "devoluciones": ["monto_devuelto"],
    "interacciones_crm": ["tiempo_respuesta_hrs"],
    "campanas_marketing": ["descuento_ofrecido"],
}
pd.DataFrame([
    {"tabla": t, "campo": c, "fuera de lo normal (IQR)": atipicos_iqr(d[t][c]), "mínimo": d[t][c].min(), "máximo": d[t][c].max()}
    for t, cols in campos.items() for c in cols
]).set_index(["tabla", "campo"])

fuera de lo normal (IQR)  mínimo  \
tabla              campo                                                    
clientes           edad                                         0   18.00   
transacciones      cantidad                                  5042    1.00   
                   precio_unitario                           2509    4.04   
                   descuento                                 2144    0.00   
                   monto_neto                                4138    3.10   
productos          costo                                       27    2.68   
                   precio_lista                                16    4.04   
                   margen_bruto_pct                             0    8.50   
devoluciones       monto_devuelto                             615    3.16   
interacciones_crm  tiempo_respuesta_hrs                       352    0.40   
campanas_marketing descuento_ofrecido                           0    0.00   

                                           máximo  
tabla              campo                           
clientes           edad                    72.000  
transacciones      cantidad                 5.000  
                   precio_unitario       1186.640  
                   descuento                0.345  
                   monto_neto            5840.920  
productos          costo                  996.200  
                   precio_lista          1186.640  
                   margen_bruto_pct        56.000  
devoluciones       monto_devuelto        5552.950  
interacciones_crm  tiempo_respuesta_hrs   518.600  
campanas_marketing descuento_ofrecido       0.250

In [4]:
# Las fechas también pueden tener valores fuera de lo normal: devoluciones por día
por_dia = dev.fecha_devolucion.value_counts().sort_index()
print(f"devoluciones por día (media): {por_dia.mean():.1f}")
print(por_dia.tail(5).to_string())
ult = dev[dev.fecha_devolucion == dev.fecha_devolucion.max()].merge(trx[["transaccion_id", "fecha"]], on="transaccion_id")
print(f"días entre compra y devolución en el último día (mediana): {(ult.fecha_devolucion - ult.fecha).dt.days.median():.0f}")

devoluciones por día (media): 9.2
fecha_devolucion
2024-12-27      8
2024-12-28      8
2024-12-29     11
2024-12-30      7
2024-12-31    158
días entre compra y devolución en el último día (mediana): 8


**Resultado:**
- **Ningún valor es imposible.** Las edades van de 18 a 72 años, los descuentos de 0% a 34,5% y las cantidades de 1 a 5.
- **Los "fuera de lo normal" son productos caros, no errores.** En precio, costo y monto, el método marca sobre todo productos de Electrónica (ver la decisión A1.7). En cantidad, marca las compras de 4 o 5 unidades, que son posibles.
- **Las fechas sí tienen un valor raro.** El 31 de diciembre de 2024 tiene 158 devoluciones; un día normal tiene 9. Son compras de diciembre devueltas después del cierre de los datos: el sistema las fechó el último día disponible. Las marcamos con `fecha_dev_recortada` (decisión A1.8). Si se cuentan las devoluciones por fecha de devolución, estas 158 inflan el último trimestre. Por eso medimos la tasa por fecha de compra.

## 2. Cruces entre tablas

Comprobamos lo que pide el enunciado (devoluciones sin una venta de origen y fechas que no tienen sentido) y además que los precios y montos cuadren.

In [5]:
def mostrar(texto, valor):
    print(f"{texto:62s} {valor}")

print("--- Claves huérfanas")
mostrar("transacciones con cliente inexistente", (~trx.cliente_id.isin(cli.cliente_id)).sum())
mostrar("transacciones con producto inexistente", (~trx.producto_id.isin(prd.producto_id)).sum())
mostrar("devoluciones sin transacción de origen", (~dev.transaccion_id.isin(trx.transaccion_id)).sum())
for nombre, t in [("devoluciones", dev), ("crm", crm), ("campañas", cam)]:
    mostrar(f"{nombre} con cliente inexistente", (~t.cliente_id.isin(cli.cliente_id)).sum())
mostrar("clientes sin ninguna transacción", (~cli.cliente_id.isin(trx.cliente_id)).sum())

print("\n--- Transacciones")
calculado = (trx.cantidad * trx.precio_unitario * (1 - trx.descuento)).round(2)
mostrar("monto_neto distinto de cantidad*precio*(1-descuento)", ((trx.monto_neto - calculado).abs() > 0.05).sum())
m = trx.merge(prd[["producto_id", "precio_lista", "costo", "categoria"]], on="producto_id")
mostrar("precio_unitario distinto del precio de lista", ((m.precio_unitario - m.precio_lista).abs() > 0.01).sum())
mostrar("venta por unidad por debajo del costo", (m.monto_neto / m.cantidad < m.costo).sum())
con_registro = trx.merge(cli[["cliente_id", "fecha_registro"]], on="cliente_id", how="left")
mostrar("compra anterior al registro del cliente", (con_registro.fecha < con_registro.fecha_registro).sum())

print("\n--- Productos")
margen_calc = ((prd.precio_lista - prd.costo) / prd.precio_lista * 100).round(1)
mostrar("margen_bruto_pct que no cuadra con costo y precio", ((margen_calc - prd.margen_bruto_pct).abs() > 0.15).sum())

print("\n--- Devoluciones")
dd = dev.merge(trx[["transaccion_id", "cliente_id", "fecha", "monto_neto"]],
               on="transaccion_id", how="left", suffixes=("", "_trx"))
mostrar("devolución antes de la compra", (dd.fecha_devolucion < dd.fecha).sum())
mostrar("se devolvió más de lo que se pagó", (dd.monto_devuelto > dd.monto_neto + 0.01).sum())
mostrar("el cliente de la devolución no es el de la compra", (dd.cliente_id != dd.cliente_id_trx).sum())
mostrar("días de la compra a la devolución (mediana / p95 / máx)",
        (dd.fecha_devolucion - dd.fecha).dt.days.quantile([0.5, 0.95, 1]).tolist())
mostrar("% de transacciones con devolución", round(len(dev) / len(trx) * 100, 2))

print("\n--- CRM")
q1, q3 = crm.tiempo_respuesta_hrs.quantile([0.25, 0.75])
umbral = q3 + 1.5 * (q3 - q1)
mostrar(f"tiempo de respuesta sobre Q3+1,5·IQR ({umbral:.1f} h)", (crm.tiempo_respuesta_hrs > umbral).sum())
mostrar("tiempo de respuesta de más de una semana", (crm.tiempo_respuesta_hrs > 168).sum())

print("\n--- Campañas")
mostrar("convertido=1 sin haber abierto", ((cam.convertido == 1) & (cam.abierto == 0)).sum())
mostrar("conversión global % / apertura %", f"{cam.convertido.mean():.2%} / {cam.abierto.mean():.2%}")

print("\n--- Segmento declarado vs comportamiento")
ultima = trx.groupby("cliente_id").fecha.max()
c = cli.set_index("cliente_id").join(ultima.rename("ultima"))
recientes = c.ultima >= trx.fecha.max() - pd.Timedelta(days=90)
mostrar("'Inactivo' con compras en los últimos 90 días", ((c.segmento == "Inactivo") & recientes).sum())

--- Claves huérfanas
transacciones con cliente inexistente                          0
transacciones con producto inexistente                         0


devoluciones sin transacción de origen                         0


devoluciones con cliente inexistente                           0
crm con cliente inexistente                                    0
campañas con cliente inexistente                               0


clientes sin ninguna transacción                               392

--- Transacciones
monto_neto distinto de cantidad*precio*(1-descuento)           0


precio_unitario distinto del precio de lista                   0
venta por unidad por debajo del costo                          711
compra anterior al registro del cliente                        24972

--- Productos
margen_bruto_pct que no cuadra con costo y precio              0

--- Devoluciones
devolución antes de la compra                                  0
se devolvió más de lo que se pagó                              0
el cliente de la devolución no es el de la compra              0
días de la compra a la devolución (mediana / p95 / máx)        [15.0, 29.0, 30.0]
% de transacciones con devolución                              13.45

--- CRM
tiempo de respuesta sobre Q3+1,5·IQR (48.9 h)                  352
tiempo de respuesta de más de una semana                       15

--- Campañas
convertido=1 sin haber abierto                                 0
conversión global % / apertura %                               9.04% / 35.93%

--- Segmento declarado vs comportamiento
'Inactivo' con

## 3. Los tres problemas más importantes

1. Hay compras anteriores a la fecha de registro del cliente. ¿Depende del canal? Por ejemplo: gente que compraba en tienda antes de registrarse en la web.
2. Hay ventas por debajo del costo. ¿De dónde salen?
3. ¿Las campañas marcadas como convertidas tienen una compra que lo confirme? ¿Hay campañas "convertidas" que se enviaron después de la compra?

In [6]:
con_registro["antes"] = con_registro.fecha < con_registro.fecha_registro
print("1) % de compras antes del registro, por canal:")
print(con_registro.groupby("canal").antes.mean().round(3).to_string())

por_cliente = con_registro.groupby("cliente_id").agg(registro=("fecha_registro", "first"), primera=("fecha", "min"))
correlacion = por_cliente.registro.map(pd.Timestamp.toordinal).corr(por_cliente.primera.map(pd.Timestamp.toordinal))
print("   correlación entre fecha de registro y primera compra:", round(correlacion, 3))

1) % de compras antes del registro, por canal:
canal
app_movil        0.492
online           0.505
tienda_fisica    0.497
   correlación entre fecha de registro y primera compra: 0.009


In [7]:
bajo_costo = m[m.monto_neto / m.cantidad < m.costo]
print("2) Ventas bajo costo:", len(bajo_costo))
print("   descuento aplicado (mín / mediana / máx):", bajo_costo.descuento.quantile([0, 0.5, 1]).round(3).tolist())
print("   por categoría:", bajo_costo.categoria.value_counts().to_dict())
print("   margen de lista de esas categorías:", prd.groupby("categoria").margen_bruto_pct.median().round(1).to_dict())

2) Ventas bajo costo: 711
   descuento aplicado (mín / mediana / máx): [0.09, 0.197, 0.345]
   por categoría: {'Electrónica': 633, 'Alimentos_y_Bebidas': 71, 'Hogar_y_Deco': 6, 'Deportes': 1}
   margen de lista de esas categorías: {'Alimentos_y_Bebidas': 27.3, 'Belleza_y_Salud': 45.4, 'Deportes': 39.2, 'Electrónica': 18.6, 'Hogar_y_Deco': 34.0, 'Ropa_y_Calzado': 42.8}


In [8]:
convertidas = cam[cam.convertido == 1]
cruce = convertidas.merge(trx[["cliente_id", "fecha"]], on="cliente_id", how="left")
cruce["dias"] = (cruce.fecha - cruce.fecha_envio).dt.days

print(f"3) {len(convertidas)} campañas marcadas como convertidas")
for ventana in (7, 14, 30):
    con_compra = cruce[cruce.dias.between(0, ventana)].campana_id.unique()
    print(f"   con compra en los {ventana:2d} días siguientes: {convertidas.campana_id.isin(con_compra).sum()}")

despues = set(cruce[cruce.dias.between(0, 30)].campana_id)
antes = set(cruce[cruce.dias.between(-30, -1)].campana_id)
print("   solo hay una compra en los 30 días ANTERIORES al envío:", len(antes - despues))

3) 797 campañas marcadas como convertidas
   con compra en los  7 días siguientes: 62
   con compra en los 14 días siguientes: 101
   con compra en los 30 días siguientes: 179
   solo hay una compra en los 30 días ANTERIORES al envío: 102


**Resultado** (el detalle está en `DECISIONES.md`):

1. **Compras antes del registro:** pasa igual en los tres canales, y la fecha de registro no tiene relación con la primera compra. La empresa tiene 12 años, pero todos los registros son de 2023 y 2024, repartidos a lo largo de los 24 meses (con más altas en el último trimestre de 2024). Una migración a un sistema nuevo habría dejado casi todas las altas al principio, y eso no pasa. No sabemos qué mide esta fecha, así que la tratamos como un campo sin información y no la usamos.
2. **Ventas bajo costo:** casi todas son de Electrónica, la categoría con menos margen. Con descuentos de entre 9% y 34,5%, la venta termina en pérdida. No es un error: es un hallazgo para A2.
3. **Campañas convertidas:** solo 179 de 797 tienen una compra en los 30 días siguientes. En 102 casos, la única compra cercana es anterior al envío (el caso que menciona el enunciado). La marca de conversión sirve para comparar tasas, pero no para calcular cuánto dinero trajo cada campaña.

## 4. Métricas de calidad de cada tabla

El enunciado pide cuatro métricas. Así las medimos:

- **Completitud:** qué porcentaje de las celdas tiene un valor real (ni vacío ni "No especificado").
- **Unicidad:** qué porcentaje de las filas tiene un código que no se repite.
- **Consistencia:** qué porcentaje de las filas cumple todas las reglas de su tabla. Debajo aparece cuántas filas incumple cada regla.
- **Oportunidad:** qué porcentaje de los registros está al día en la fecha de corte (31 de diciembre de 2024). En clientes, devoluciones y atención al cliente, que la fecha caiga dentro de los 24 meses del estudio. En ventas y campañas, además, que haya pasado suficiente tiempo para que el dato esté completo. Por ejemplo, una compra del 28 de diciembre todavía podía devolverse después del 31, así que no sabemos si se devolvió.

In [9]:
inicio, fin = pd.Timestamp("2023-01-01"), pd.Timestamp("2024-12-31")

def completitud(nombre):
    t = pd.read_csv(RAW / f"{nombre}.csv", dtype=str, keep_default_na=False)
    faltan = t.apply(lambda s: (s.str.strip() == "") | s.str.strip().str.lower().isin(NULOS_DISFRAZADOS))
    return 100 * (1 - faltan.values.mean())

ult_compra = trx.groupby("cliente_id").fecha.max()
cli_ult = cli.cliente_id.map(ult_compra)
precio_ok = trx.merge(prd[["producto_id", "precio_lista"]], on="producto_id", how="left").set_index(trx.index)

reglas = {
    "clientes": {
        "edad entre 18 y 100": cli.edad.between(18, 100),
        "género informado": cli.genero != "No especificado",
        "'Inactivo' sin compras en los últimos 90 días": ~((cli.segmento == "Inactivo") & (cli_ult >= fin - pd.Timedelta(days=90))),
    },
    "transacciones": {
        "cliente existe": trx.cliente_id.isin(cli.cliente_id),
        "producto existe": trx.producto_id.isin(prd.producto_id),
        "monto_neto cuadra": (trx.monto_neto - calculado).abs() <= 0.05,
        "precio = precio de lista": (precio_ok.precio_unitario - precio_ok.precio_lista).abs() <= 0.01,
        "compra posterior al registro": ~con_registro.set_index(trx.index).antes,
    },
    "productos": {
        "costo < precio de lista": prd.costo < prd.precio_lista,
        "margen_bruto_pct cuadra": (margen_calc - prd.margen_bruto_pct).abs() <= 0.15,
    },
    "devoluciones": {
        "transacción de origen existe": dev.transaccion_id.isin(trx.transaccion_id),
        "mismo cliente que la compra": (dd.cliente_id == dd.cliente_id_trx).values,
        "después de la compra": (dd.fecha_devolucion >= dd.fecha).values,
        "no devuelve más de lo pagado": (dd.monto_devuelto <= dd.monto_neto + 0.01).values,
    },
    "interacciones_crm": {
        "cliente existe": crm.cliente_id.isin(cli.cliente_id),
        "tiempo de respuesta ≥ 0": crm.tiempo_respuesta_hrs >= 0,
        "respuesta en menos de una semana": crm.tiempo_respuesta_hrs <= 168,
    },
    "campanas_marketing": {
        "cliente existe": cam.cliente_id.isin(cli.cliente_id),
        "convertida implica abierta": ~((cam.convertido == 1) & (cam.abierto == 0)),
        "conversión con compra en 30 días": ~((cam.convertido == 1) & ~cam.campana_id.isin(despues)),
    },
}

# Oportunidad: fecha dentro del período y, donde aplica, con tiempo suficiente para que el dato esté completo
p95_devolucion = int((dd.fecha_devolucion - dd.fecha).dt.days.quantile(0.95))
oportunidad = {
    "clientes": cli.fecha_registro.between(inicio, fin),
    "transacciones": trx.fecha <= fin - pd.Timedelta(days=p95_devolucion),
    "productos": pd.Series(True, index=prd.index),  # no tiene fecha
    "devoluciones": dev.fecha_devolucion.between(inicio, fin),
    "interacciones_crm": crm.fecha.between(inicio, fin),
    "campanas_marketing": cam.fecha_envio <= fin - pd.Timedelta(days=30),
}

filas = []
for nombre, t in d.items():
    ok = pd.concat({k: pd.Series(np.asarray(v), index=t.index) for k, v in reglas[nombre].items()}, axis=1)
    filas.append({
        "dataset": nombre,
        "filas": len(t),
        "completitud %": round(completitud(nombre), 2),
        "unicidad %": round(100 * (1 - t.iloc[:, 0].duplicated().mean()), 2),
        "consistencia %": round(100 * ok.all(axis=1).mean(), 2),
        "oportunidad %": round(100 * oportunidad[nombre].mean(), 2),
    })
    print(f"\n{nombre}")
    for regla, cumple in ok.items():
        print(f"   {regla:48s} incumplen: {(~cumple).sum()}")

calidad = pd.DataFrame(filas).set_index("dataset")
calidad


clientes
   edad entre 18 y 100                              incumplen: 0
   género informado                                 incumplen: 433
   'Inactivo' sin compras en los últimos 90 días    incumplen: 108

transacciones
   cliente existe                                   incumplen: 0
   producto existe                                  incumplen: 0
   monto_neto cuadra                                incumplen: 0
   precio = precio de lista                         incumplen: 0
   compra posterior al registro                     incumplen: 24972

productos
   costo < precio de lista                          incumplen: 0
   margen_bruto_pct cuadra                          incumplen: 0

devoluciones
   transacción de origen existe                     incumplen: 0
   mismo cliente que la compra                      incumplen: 0
   después de la compra                             incumplen: 0
   no devuelve más de lo pagado                     incumplen: 0

interacciones_crm
   cliente ex

,filas,completitud %,unicidad %,consistencia %,oportunidad %
dataset,,,,,
clientes,8000,99.23,100.0,93.35,100.00
transacciones,50000,100.00,100.0,50.06,96.02
productos,318,100.00,100.0,100.00,100.00
devoluciones,6724,100.00,100.0,100.00,100.00
interacciones_crm,4684,100.00,100.0,99.68,100.00
campanas_marketing,8820,100.00,100.0,92.99,95.94


In [10]:
# La regla "compra posterior al registro" depende de una fecha que no usamos (A1.1). Así queda la consistencia sin ella:
sin_registro = {k: v for k, v in reglas["transacciones"].items() if k != "compra posterior al registro"}
ok = pd.concat({k: pd.Series(np.asarray(v), index=trx.index) for k, v in sin_registro.items()}, axis=1)
print(f"consistencia de transacciones con la regla de registro: {calidad.loc['transacciones', 'consistencia %']}% | sin ella: {100 * ok.all(axis=1).mean():.2f}%")

consistencia de transacciones con la regla de registro: 50.06% | sin ella: 100.00%


**Cómo leer la tabla:**

- La completitud y la unicidad están casi al 100%. Los datos no tienen problemas de forma; los problemas están en lo que significan.
- La consistencia de ventas baja a 50% por una sola regla: las compras anteriores al registro. Como esa fecha no tiene información (A1.1), damos también la consistencia sin esa regla: 100%.
- La consistencia de campañas baja por las conversiones que no tienen una compra que las confirme.
- En oportunidad, las ventas de los últimos 29 días cuentan como incompletas, porque todavía podían recibir una devolución. Las campañas del último mes también, porque no pasaron los 30 días para convertir. En la práctica el efecto es pequeño: diciembre tiene 13,4% de devoluciones, igual que el resto del año. Aun así, hay que tenerlo presente al mirar los últimos meses.

In [11]:
# Comprobación de lo anterior: tasa de devolución por mes de compra
tasa_mes = trx.assign(devuelta=trx.transaccion_id.isin(dev.transaccion_id)).groupby(trx.fecha.dt.to_period("M")).devuelta.mean() * 100
print(f"p95 de días hasta la devolución: {p95_devolucion}")
print(tasa_mes.tail(4).round(1).to_string())
print(f"promedio del resto del período: {tasa_mes.iloc[:-2].mean():.1f}%")

p95 de días hasta la devolución: 29
fecha
2024-09    13.6
2024-10    14.5
2024-11    12.4
2024-12    13.4
Freq: M
promedio del resto del período: 13.5%


## 5. ¿Los datos dicen lo mismo que el enunciado?

El enunciado da tres cifras como punto de partida del caso. Antes de construir la historia sobre ellas, las comprobamos. Así medimos cada una:

- **Ticket promedio:** cuánto paga un cliente, en promedio, en cada compra. Es el total vendido dividido entre el número de compras. El enunciado dice que bajó 8% "interanual". Lo comparamos de dos formas: 2024 contra 2023 completos, y el último trimestre de 2024 contra el mismo trimestre de 2023.
- **Tasa de devoluciones:** de cada 100 ventas de un trimestre, cuántas se devolvieron. El enunciado dice que "aumentó al 14%" en el último trimestre.
- **Conversión de campañas:** de cada 100 campañas enviadas en un trimestre, cuántas quedaron marcadas como convertidas (el cliente compró). El enunciado dice que bajó de 22% a 11% en 18 meses.

In [12]:
por_anio = trx.groupby(trx.fecha.dt.year).monto_neto.mean()
print(f"Ticket promedio 2023: {por_anio[2023]:.2f} | 2024: {por_anio[2024]:.2f} | cambio: {100 * (por_anio[2024] / por_anio[2023] - 1):.1f}%")

trx_q = trx.fecha.dt.to_period("Q")
ticket_q = trx.groupby(trx_q).monto_neto.mean()
print(f"Ticket del último trimestre 2023: {ticket_q.iloc[3]:.2f} | 2024: {ticket_q.iloc[-1]:.2f} | cambio: {100 * (ticket_q.iloc[-1] / ticket_q.iloc[3] - 1):.1f}%")

primeros_6 = cam[cam.fecha_envio < "2023-07-01"].convertido.mean() * 100
ultimos_6 = cam[cam.fecha_envio >= "2024-07-01"].convertido.mean() * 100
print(f"Conversión primeros 6 meses: {primeros_6:.1f}% | últimos 6 meses: {ultimos_6:.1f}%")
print(f"Ventas por canal (%): {(trx.canal.value_counts(normalize=True) * 100).round(1).to_dict()}")

# Devoluciones de cuatro formas: contando ventas o sumando dinero, y por fecha de compra o de devolución
dd5 = dev.merge(trx[["transaccion_id", "fecha"]], on="transaccion_id")
q_compra, q_dev = dd5.fecha.dt.to_period("Q"), dd5.fecha_devolucion.dt.to_period("Q")
recortada = dd5.fecha_devolucion == dd5.fecha_devolucion.max()
ventas_q, monto_q = trx.groupby(trx_q).size(), trx.groupby(trx_q).monto_neto.sum()

# Margen bruto % (costo del catálogo × cantidad)
costo = trx.merge(prd[["producto_id", "costo"]], on="producto_id").set_index(trx.index)
margen_q = (trx.monto_neto - costo.costo * trx.cantidad).groupby(trx_q).sum() / monto_q * 100
margen_a = (trx.monto_neto - costo.costo * trx.cantidad).groupby(trx.fecha.dt.year).sum() / trx.groupby(trx.fecha.dt.year).monto_neto.sum() * 100
print(f"Margen bruto 2023: {margen_a[2023]:.1f}% | 2024: {margen_a[2024]:.1f}% | ventas 2023: {trx[trx.fecha.dt.year == 2023].monto_neto.sum():,.0f} | 2024: {trx[trx.fecha.dt.year == 2024].monto_neto.sum():,.0f}")

conversion = cam.groupby(cam.fecha_envio.dt.to_period("Q")).convertido.mean() * 100
pd.DataFrame({
    "ticket promedio": ticket_q.round(1),
    "devol. % ventas (fecha compra)": (q_compra.value_counts() / ventas_q * 100).round(1),
    "devol. % dinero (fecha compra)": (dd5.groupby(q_compra).monto_devuelto.sum() / monto_q * 100).round(1),
    "devol. % ventas (fecha devolución)": (q_dev.value_counts() / ventas_q * 100).round(1),
    "ídem sin el 31-dic": (q_dev[~recortada].value_counts() / ventas_q * 100).round(1),
    "margen bruto %": margen_q.round(1),
    "conversión %": conversion.round(1),
})

Ticket promedio 2023: 351.29 | 2024: 344.49 | cambio: -1.9%
Ticket del último trimestre 2023: 347.69 | 2024: 347.02 | cambio: -0.2%
Conversión primeros 6 meses: 11.8% | últimos 6 meses: 6.4%
Ventas por canal (%): {'tienda_fisica': 44.7, 'online': 38.4, 'app_movil': 16.9}


Margen bruto 2023: 24.7% | 2024: 24.9% | ventas 2023: 8,708,407 | 2024: 8,684,581


,ticket promedio,devol. % ventas (fecha compra),devol. % dinero (fecha compra),devol. % ventas (fecha devolución),ídem sin el 31-dic,margen bruto %,conversión %
2023Q1,350.4,13.2,14.8,10.8,10.8,24.6,11.3
2023Q2,344.0,13.6,15.3,13.5,13.5,24.7,12.3
2023Q3,363.2,13.0,13.0,13.4,13.4,24.7,11.2
2023Q4,347.7,13.8,14.1,13.9,13.9,24.6,10.2
2024Q1,340.4,13.3,15.8,13.1,13.1,25.4,7.9
2024Q2,341.7,13.3,14.1,13.2,13.2,24.9,6.8
2024Q3,348.9,13.8,14.5,13.9,13.9,24.7,7.1
2024Q4,347.0,13.5,15.1,15.8,13.3,24.7,5.6


**Resultado:** ninguna de las tres cifras se reproduce tal como la cuenta el enunciado. Para cada una probamos más de una definición, por si el enunciado midió de otra forma.

| Métrica | Qué dice el enunciado | Qué muestran los datos | ¿Coincide? |
|---|---|---|---|
| Ticket promedio | Bajó 8% en un año | En 2023 cada compra promedió 351 y en 2024, 344: bajó 1,9%. Si comparamos solo el último trimestre de cada año, bajó 0,2% | **No.** El gasto por compra casi no cambió |
| Devoluciones | Subieron hasta 14% en el último trimestre | Contando ventas, siempre entre 13,0% y 13,8%. Sumando dinero, siempre entre 13% y 16% (14,6% de media): ese es el "14%" del enunciado. Por fecha de devolución, el último trimestre sube a 15,8%, pero solo por las 158 devoluciones fechadas el 31-dic (sin ellas, 13,3%) | **El nivel sí, la subida no.** Las devoluciones son altas desde siempre |
| Conversión de campañas | Bajó de 22% a 11% en 18 meses | Bajó de 11,3% (primer trimestre de 2023) a 5,6% (último trimestre de 2024) | **En parte.** Se cayó a la mitad, pero empezó en 11%, no en 22% |

Agregamos una cuarta medida que el enunciado da por hecha, la "presión en los márgenes". **El margen bruto no cae: 24,7% en 2023 y 24,9% en 2024, con ventas casi iguales (8,71 y 8,68 millones).**

Además, hay un canal de venta que el enunciado no menciona: la app móvil, con el 17% de las ventas.

**Qué significa para la historia:** la empresa no está empeorando. El margen, el ticket y las devoluciones están estables desde hace dos años. Lo único que cae es la conversión registrada de las campañas (en A4 vemos que esa caída no aparece en las compras). El problema no es una caída, sino un nivel: la empresa lleva dos años perdiendo margen siempre en los mismos sitios. Las fases A2 a A4 buscan cuáles son.

## 6. Cómo se relacionan las tablas

El diagrama está en `docs/modelo.md`. Es una **constelación de hechos**: cuatro tablas de hechos que comparten las mismas dimensiones (cliente, producto y fecha). Cada hecho tiene su propio nivel de detalle:

| Hecho | Una fila es… | Dimensiones |
|---|---|---|
| Ventas (`transacciones`) | un producto que un cliente compró en una fecha | cliente, producto, fecha |
| Devoluciones | la devolución de una venta (como mucho una por venta) | venta, cliente, fecha de devolución (y el producto a través de la venta) |
| Campañas | un envío de campaña a un cliente | cliente, fecha de envío |
| Atención al cliente (`interacciones_crm`) | un contacto de un cliente | cliente, fecha |

Campañas y atención al cliente solo se conectan con el cliente, no con una venta. Por eso no sabemos qué campaña generó qué compra y hay que deducirlo por las fechas.

Aquí comprobamos cuántos registros tiene cada relación:

In [13]:
print("devoluciones por transacción (máx):", dev.transaccion_id.value_counts().max())
print("¿cada devolución apunta a una sola transacción?", dev.devolucion_id.is_unique)
print("transacciones por cliente (mediana / máx):", trx.cliente_id.value_counts().median(), trx.cliente_id.value_counts().max())
print("campañas por cliente (mediana / máx):", cam.cliente_id.value_counts().median(), cam.cliente_id.value_counts().max())
print("interacciones por cliente (mediana / máx):", crm.cliente_id.value_counts().median(), crm.cliente_id.value_counts().max())

devoluciones por transacción (máx): 1
¿cada devolución apunta a una sola transacción? True
transacciones por cliente (mediana / máx): 4.0 34
campañas por cliente (mediana / máx): 2.0 4
interacciones por cliente (mediana / máx): 2.0 7


## 7. Comprobación de los datos limpios

Comprobamos que los datos limpios tienen las mismas filas que los originales y que las marcas coinciden con lo que contamos arriba. También calculamos las cifras que usamos en `DECISIONES.md` para justificar los umbrales.

In [14]:
limpio = cargar_limpio(recalcular=True)

for nombre in d:
    assert len(limpio[nombre]) == len(d[nombre]), nombre

t, c, k = limpio["transacciones"], limpio["clientes"], limpio["campanas_marketing"]
print("ventas bajo costo:", t.bajo_costo.sum())
print("outliers de monto por categoría:", t.outlier_cat.sum())
print("clientes sin compras:", c.sin_compras.sum())
print("conversiones respaldadas:", k.conversion_respaldada.sum(), "de", (k.convertido == 1).sum())
print("CRM con respuesta atípica:", limpio["interacciones_crm"].respuesta_atipica.sum())

# Datos que respaldan los umbrales de DECISIONES.md
entre_compras = trx.sort_values("fecha").groupby("cliente_id").fecha.diff().dt.days
print("días entre compras (mediana / p75 / p90):", entre_compras.quantile([0.5, 0.75, 0.9]).tolist())
q1, q3 = trx.monto_neto.quantile([0.25, 0.75])
global_iqr = trx.monto_neto > q3 + 1.5 * (q3 - q1)
print("outliers de monto con IQR global:", global_iqr.sum(), "| de Electrónica:", (global_iqr & t.categoria.eq("Electrónica")).sum())
print("clientes sin compras que reciben campañas:", c[c.sin_compras].cliente_id.isin(k.cliente_id).sum())
conv = k[k.convertido == 1]
print("% de conversiones respaldadas por tipo de campaña:")
print((conv.groupby("tipo_campana").conversion_respaldada.mean() * 100).round(1).to_string())
print("% respaldadas por año:", (conv.groupby(conv.fecha_envio.dt.year).conversion_respaldada.mean() * 100).round(1).to_dict())
print(f"conversión si solo contáramos las respaldadas: {k.conversion_respaldada.mean():.1%}")

# ¿Los que llegan a 180 días sin comprar vuelven? Solo casos con 180 días más de datos para observarlo
orden = trx.sort_values(["cliente_id", "fecha"]).copy()
orden["siguiente"] = orden.groupby("cliente_id").fecha.shift(-1)
orden["dia_180"] = orden.fecha + pd.Timedelta(days=180)
observables = orden[orden.dia_180 <= trx.fecha.max() - pd.Timedelta(days=180)]
llegan_a_180 = observables[observables.siguiente.isna() | (observables.siguiente > observables.dia_180)]
vuelven = llegan_a_180.siguiente <= llegan_a_180.dia_180 + pd.Timedelta(days=180)
print(f"clientes que llegan a 180 días sin comprar: {len(llegan_a_180)} casos; vuelven en los 180 días siguientes: {vuelven.mean():.1%}")
print(f"margen bruto total: {t.margen_bruto.sum():,.0f} sobre ventas de {t.monto_neto.sum():,.0f} ({t.margen_bruto.sum() / t.monto_neto.sum():.1%})")
pd.crosstab(c.segmento, c.segmento_rfm, margins=True)

ventas bajo costo: 711
outliers de monto por categoría: 3420
clientes sin compras: 392
conversiones respaldadas: 179 de 797
CRM con respuesta atípica: 352
días entre compras (mediana / p75 / p90): [40.0, 87.0, 162.0]
outliers de monto con IQR global: 4138 | de Electrónica: 2672
clientes sin compras que reciben campañas: 242
% de conversiones respaldadas por tipo de campaña:
tipo_campana
email_descuento       24.5
email_fidelizacion    24.7
push_reactivacion     21.0
sms_oferta_flash      19.6
social_ads            17.3
% respaldadas por año: {2023: 22.2, 2024: 22.9}
conversión si solo contáramos las respaldadas: 2.0%
clientes que llegan a 180 días sin comprar: 4105 casos; vuelven en los 180 días siguientes: 52.1%
margen bruto total: 4,310,329 sobre ventas de 17,392,987 (24.8%)


segmento_rfm,Esporádico,Frecuente,Inactivo,Nuevo,Premium,Sin compras,All
segmento,,,,,,,
Esporádico,1408,0,1324,48,4,43,2827
Frecuente,319,1629,170,0,304,0,2422
Inactivo,119,0,654,108,0,255,1136
Nuevo,169,0,488,59,0,94,810
Premium,11,345,11,0,438,0,805
All,2026,1974,2647,215,746,392,8000


In [15]:
# ¿Qué mide el segmento que trae el archivo? Compras de cada cliente en los 24 meses, por segmento
c.groupby("segmento").n_compras.describe()[["count", "min", "50%", "max"]].astype(int)

,count,min,50%,max
segmento,,,,
Esporádico,2827,0,3,6
Frecuente,2422,1,11,24
Inactivo,1136,0,1,1
Nuevo,810,0,1,3
Premium,805,4,18,34


El 52% de los clientes que llegan a 180 días sin comprar vuelve después. Por eso el segmento calculado "Inactivo" significa "en riesgo", no "perdido".

**Qué es el segmento del archivo:** la tabla de arriba muestra que cada segmento es una franja del número de compras del cliente en los **24 meses completos**: los "Inactivo" tienen 0 o 1 compra, los "Nuevo" hasta 3, los "Esporádico" hasta 6, y los "Frecuente" y "Premium" tienen más. Es decir, el segmento se asignó mirando lo que el cliente hizo hasta el final de los datos. Coincide con el comportamiento en **frecuencia**, pero no en **recencia**: muchos "Esporádico" y "Nuevo" llevan más de 180 días sin comprar (tabla del final).

Esto tiene una consecuencia importante: el segmento del archivo es un **resultado** del comportamiento, no algo que se supiera de antemano. Sirve para *describir* (el enunciado pide resultados "por segmento"), pero no para *predecir*. Decir "los Inactivo nunca vuelven" sería una tautología, porque se llaman Inactivo justamente por no haber vuelto. En A3, la probabilidad de que un cliente vuelva se estima con datos conocidos en la fecha de corte, no con este segmento (decisión A1.4).